# 환경 설정 및 준비

`(1) Env 환경변수`

In [ ]:
from dotenv import load_dotenv
load_dotenv()

`(2) 기본 라이브러리`

In [ ]:
import os
from glob import glob

from pprint import pprint
import json

`(3) LLM 설정`

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.1,
    top_p=0.9, 
)

# **문서 전처리 파이프라인**

* 문서 전처리의 첫 단계는 데이터 정제로, 원본 문서에서 불필요한 요소(HTML 태그, 특수문자, 중복 내용 등)를 제거하고 텍스트를 표준화하는 과정입니다. 이는 검색 품질과 직결되는 중요한 단계입니다.

* 문서 청킹(Chunking)은 긴 문서를 의미 있는 작은 단위로 분할하는 과정으로, 문장 단위나 단락 단위로 나누되 문맥의 연속성을 유지하는 것이 핵심입니다. 이는 검색 정확도와 답변 생성의 품질에 직접적인 영향을 미칩니다.

* 임베딩(Embedding) 생성은 텍스트를 고차원의 벡터로 변환하는 과정으로, 문서의 의미적 특성을 수치화하여 효율적인 검색을 가능하게 합니다. 이때 사용되는 임베딩 모델의 선택이 검색 성능을 좌우하는 중요한 요소가 됩니다.

* 마지막으로 벡터 데이터베이스 색인화 단계에서는 생성된 임베딩을 효율적으로 저장하고 검색할 수 있는 구조로 변환합니다. 이는 대규모 문서 집합에서도 빠른 검색을 가능하게 하는 핵심 요소입니다.


### 1) 문서 로드

- 국토교통부 주택청약 FAQ에서 일부 내용(청약자격, 청약통장)을 발췌하여 재가공
- Q1 ~ Q50까지 모두 50개의 문답이 포함된 텍스트 파일

In [ ]:
# 파일 경로 설정
faq_text_file = "../data/housing_faq.txt"

# 파일 읽기 - 파이썬 내장 함수 사용
with open(faq_text_file, 'r', encoding='utf-8') as f:
    faq_text = f.read()

# 파일 내용 확인
print(faq_text[:500])

##### ***[실습] TextLoader를 사용하여, 텍스트 문서를 로드합니다.*** 

In [ ]:
# 여기에 코드를 작성하세요.
from langchain_community.document_loaders import TextLoader

# TextLoader 클래스를 사용하여 FAQ 텍스트 파일을 로드
loader = TextLoader(faq_text_file, 'utf-8')
docs = loader.load()
len(docs)

In [ ]:
# 문서 확인
print(docs[0].page_content[:500])

In [ ]:
# 문서 메타데이터 확인
docs[0].metadata

### 2) 문서 전처리

`(1) 각 질문과 답변을 쌍으로 추출하여 정리 (정규표현식 활용)`

In [ ]:
import re

def extract_qa_pairs(text):
    qa_pairs = []
    
    # 텍스트를 라인별로 분리하고 각 라인의 앞뒤 공백 제거
    lines = [line.strip() for line in text.split('\n')]
    current_question = None
    current_answer = []
    current_number = None
    in_answer = False
    
    for i, line in enumerate(lines):
        if not line:  # 빈 라인 처리
            if in_answer and current_answer and i + 1 < len(lines) and lines[i + 1].startswith('Q'):
                # 다음 질문이 시작되기 전 빈 줄이면 현재 QA 쌍 저장
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
                in_answer = False
                current_answer = []
            continue
            
        # 새로운 질문 확인 (Q 다음에 숫자가 오는 패턴)
        q_match = re.match(r'Q(\d+)\s+(.*)', line)
        if q_match:
            # 이전 QA 쌍이 있으면 저장
            if current_question is not None and current_answer:
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
            
            # 새로운 질문 시작
            current_number = int(q_match.group(1))
            current_question = q_match.group(2).strip().rstrip('?') + '?'  # 질문 마크 정규화
            current_answer = []
            in_answer = False
            
        # 답변 시작 확인
        elif line.startswith('A ') or (current_question and not current_answer and line):
            in_answer = True
            current_answer.append(line.lstrip('A '))
            
        # 기존 답변에 내용 추가
        elif current_question is not None and (in_answer or not line.startswith('Q')):
            if in_answer or (current_answer and not line.startswith('Q')):
                current_answer.append(line)
    
    # 마지막 QA 쌍 처리
    if current_question is not None and current_answer:
        qa_pairs.append({
            'number': current_number,
            'question': current_question,
            'answer': ' '.join(current_answer).strip()
        })
    
    # 번호 순서대로 정렬
    qa_pairs.sort(key=lambda x: x['number'])
    
    return qa_pairs

In [ ]:
# QA 쌍 추출
qa_pairs = extract_qa_pairs(docs[0].page_content) 

print(f"추출된 QA 쌍 개수: {len(qa_pairs)}")
print(f"추출된 첫번째 QA: \n{qa_pairs[0]}")

`(2) LLM으로 추가 정보를 추출`
- 텍스트에서 키워드와 핵심 개념을 추출하는 체인
- 메타데이터 or 본문(page_content)에 추가하여 검색에 활용    

##### ***[실습] 다음 모델 스키마를 사용하여, 키워드와 요약을 추출하는 체인을 완성합니다.*** 

In [ ]:
# QA 쌍 확인 (첫번째)
print((qa_pairs[0]['question']+"\n\n"+qa_pairs[0]['answer']))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from typing import List

# 출력 형식 정의
class KeywordOutput(BaseModel):
    keyword: str = Field(description="텍스트에서 추출한 가장 중요한 키워드(법률용어, 주제 등))")
    summary: str = Field(description="텍스트의 간단한 요약")

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 텍스트 분석 전문가입니다. 
주어진 텍스트에서 중요한 키워드를 추출하고, 텍스트의 간단한 요약을 작성하는 것이 당신의 역할입니다.

## 추출 지침:
- 텍스트의 맥락을 고려하여 핵심 용어나 전문 용어를 추출합니다
- 주요 아이디어나 원리, 개념을 포함합니다
- 가장 중요한 키워드를 1개 추출합니다
- 요약은 1문장으로 간결하게 작성합니다

## 출력 형식:
- keyword: 가장 중요한 키워드 
- summary: 텍스트의 간단한 요약"""),
    
    ("user", "다음 텍스트를 분석해주세요:\n\n{input_text}")
])

# LCEL 체인 구성
llm_with_structure = llm.with_structured_output(KeywordOutput) 
keyword_extractor = prompt | llm_with_structure

# 텍스트 추출 테스트     
result = keyword_extractor.invoke(qa_pairs[0]['question']+qa_pairs[0]['answer'])
print("키워드:", result.keyword)
print("요약:", result.summary)

`(3) QA 쌍을 문자열 포맷팅하고 문서 객체로 변환`

In [ ]:
from langchain_core.documents import Document

def format_qa_pairs(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 문서 객체로 변환
    """
    processed_docs = []
    for pair in qa_pairs:

        # QA 쌍을 포맷팅
        formatted_output = (
            f"[{pair['number']}]\n"
            f"질문: {pair['question']}\n"
            f"답변: {pair['answer']}\n"
        )

        # 키워드와 요약 추출
        result = keyowrd_extractor.invoke(pair['question']+"\n\n"+pair['answer'])

        # 문서 객체 생성
        doc = Document(
            page_content=formatted_output,
            metadata={
                'question_id': int(pair['number']),
                'question': pair['question'],
                'answer': pair['answer'],
                'keyword': result.keywords,
                'summary': result.summary
            }
        )
        processed_docs.append(doc)

    return processed_docs


# QA 쌍 포맷팅
formatted_docs = format_qa_pairs(qa_pairs)
print(f"포맷팅된 문서 개수: {len(formatted_docs)}")

# 문서 확인
print(formatted_docs[0].page_content)
print("-" * 200)
# 문서 메타데이터 확인
pprint(formatted_docs[0].metadata)

In [ ]:
# 문서 저장
output_file = "../data/housing_faq_formatted.json"

with open(output_file, 'w', encoding='utf-8-sig') as f:
    json.dump([doc.model_dump() for doc in formatted_docs], f, indent=2, ensure_ascii=False)  # 한글이 유니코드로 변환되지 않도록 설정
print(f"포맷팅된 문서를 {output_file}에 저장했습니다.")

##### ***[실습] 문서 객체를 포맷팅하여 구성합니다.*** 

- 요약문을 시맨틱 검색에 활용합니다. 다음 구조로 문서 객체를 생성합니다. 
    - page_content: 요약
    - metadata: 기타 정보

In [ ]:
# 여기에 코드를 작성하세요.


def format_qa_pairs_with_summary(qa_pairs):
    """
    추출된 QA 쌍을 포맷팅하여 문서 객체로 변환
    """
    processed_docs = []
    for pair in qa_pairs:

        # 키워드와 요약 추출
        result = keyword_extractor.invoke(pair['question']+"\n\n"+pair['answer'])

        # 문서 객체 생성
        doc = Document(
            page_content=result.summary,
            metadata={
                'question_id': int(pair['number']),
                'question': pair['question'],
                'answer': pair['answer'],
                'keyword': result.keyword,
            }
        )
        processed_docs.append(doc)

    return processed_docs


In [ ]:
# QA 쌍 포맷팅
summary_formatted_docs = format_qa_pairs_with_summary(qa_pairs) 
print(f"포맷팅된 문서 개수: {len(summary_formatted_docs)}")

# 문서 확인
print(summary_formatted_docs[0].page_content)
print("-" * 200)
# 문서 메타데이터 확인
pprint(summary_formatted_docs[0].metadata)

# 벡터 저장 

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 문서 벡터 저장
vector_store = Chroma.from_documents(  
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db",
    persist_directory="./chroma_db",
)

##### ***[실습] 요약 문서(summary_formatted_docs)를 벡터 스토어에 저장합니다.*** 

- OpenAI (text-embedding-3-small) 임베딩 모델 사용
- Chromda DB 사용

In [ ]:
# 여기에 코드를 작성하세요.
vector_store._collection.count()

# 문서 검색

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
vector_store = Chroma(
    collection_name="housing_faq_db",
    persist_directory="./chroma_db", 
    embedding_function=embeddings,
)

##### ***[실습] 앞에서 저장한 벡터 스토어를 로드합니다.*** 

- OpenAI (text-embedding-3-small) 임베딩 모델 사용
- Chromda DB 사용

In [ ]:
# 여기에 코드를 작성하세요.


# 벡터 저장소 로드
vector_store_with_summary = None

In [ ]:
# 검색기 생성 - 유사도 기반 상위 3개 문서 검색
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3},
)

# 테스트 질문
query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

##### ***[실습] MMR 검색기를 정의합니다.*** 

- 요약 문서 벡터 스토어를 사용
- 10개의 문서를 가져와서, 다양성 기반으로 3개를 선택 (다양성은 중간 수준 적용)

In [ ]:
# 여기에 코드를 작성하세요.

mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": 0.5},
)

# 테스트 질문
query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = mmr_retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print(result.metadata['question'])
    print(result.metadata['answer'])
    print("=" * 50)

### **[심화] 메타데이터 기반 필터링**

- Chroma 문서: https://docs.trychroma.com/docs/querying-collections/metadata-filtering

In [ ]:
# 단일 필드 정확히 일치
retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": "주택건설지역"}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# $eq 연산자 사용 - 정확히 일치

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$eq": "주택건설지역"}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# $ne (Not Equal) 연산자 사용 - 정확히 일치하지 않는 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$ne": "주택건설지역"}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# $in 연산자로 여러 값 중 일치하는 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"keyword": {"$in": ["주택건설지역", "청약예금"]}}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# 숫자 범위 검색 ($gt, $gte, $lt, $lte) - question_id가 10 이상인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"question_id": {"$gte": 10}}},
)

query = "무주택자 기준은 무엇인가요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# $and로 여러 조건 조합 - keyword가 "주택건설지역"이고 question_id가 10 미만인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"$and": [
        {"keyword": "주택건설지역"}, 
        {"question_id": {"$lt": 10}}
    ]}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# $or로 여러 조건 중 하나 일치하는 문서 검색 - keyword가 "주택건설지역"이거나 question_id가 10 이상인 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={"filter": {"$or": [
        {"keyword": "주택건설지역"}, 
        {"question_id": {"$gte": 10}}
    ]}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# 정규식 패턴 매칭 - page_content 본문에 "주택건설지역"이 포함된 문서 검색

retriever = vector_store.as_retriever(
    search_kwargs={'where_document': {'$contains': '주택건설지역'}},
)

query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

##### ***[실습] 메타데이터 필터링 조건을 적용하는 실습을 수행합니다..*** 

- 요약 문서 벡터 스토어 기반 MMR 검색기에 적용 

In [ ]:
# 여기에 코드를 작성하세요.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Optional

class MetadataFilter(BaseModel):
    keyword: Optional[str] = Field(description="검색할 키워드")
    keyword_expression: Optional[str] = Field(description="키워드 검색 표현식")
    lower_question_id: Optional[int] = Field(description="질문 ID의 최소값")
    upper_question_id: Optional[int] = Field(description="질문 ID의 최대값")
    lower_question_id_expression: Optional[str] = Field(description="질문 ID 최소값에 적용할 검색 표현식(gt, gte)")
    upper_question_id_expression: Optional[str] = Field(description="질문 ID 최대값에 적용할 검색 표현식(lt, lte)")
    and_or_expression: Optional[int] = Field(description="AND/OR 검색 표현식(and, or)")


system_prompt = """사용자 쿼리에서 키워드와 질문 ID 정보를 추출하여 Chroma DB 검색 필터를 생성한다.
다음 예시를 참조한다:

1. 키워드 검색 예시:
- 입력: "주택건설 관련 문서 찾아줘"
출력:
keyword: "주택건설"
keyword_expression: "$eq"

2. 질문 ID 검색 예시:
- 입력: "질문 ID 10번 이상인 문서"
출력:
lower_question_id: 10
lower_question_id_expression: "$gte"

3. 복합 검색 예시:
- 입력: "주택건설 키워드가 있으면서 질문 ID가 5번에서 15번 사이인 문서"
출력:
keyword: "주택건설"
keyword_expression: "$eq"
question_id: 5 
question_id_expression: "$gte"

검색 표현식은 다음과 같이 사용한다:
- 동등 비교: $eq
- 크거나 같음: $gte
- 작거나 같음: $lte
- 범위 검색: $gt, $lt

요청에 해당 정보가 없는 경우 해당 필드는 null로 반환한다.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{query}")
])

# 구조화된 출력을 위한 체인 생성
model_with_structure = llm.with_structured_output(MetadataFilter)
metadata_chain = prompt | model_with_structure

# 사용 예시
query = "주택건설지역 관련 문서를 10번 이하인 문서중에서 검색해주세요"
filter_params = metadata_chain.invoke({"query": query})

print(f"키워드: {filter_params.keyword}")
print(f"키워드 표현식: {filter_params.keyword_expression}")
print(f"질문 ID: {filter_params.question_id}")
print(f"질문 ID 표현식: {filter_params.question_id_expression}")

In [ ]:
# 메타데이터 필터 생성
filter_dict = {}

if filter_params.keyword and filter_params.question_id:
    # 두 조건 모두 있는 경우 AND 연산자 사용
    filter_dict = {
        "$and": [
            {"keyword": {filter_params.keyword_expression: filter_params.keyword}},
            {"question_id": {filter_params.question_id_expression: filter_params.question_id}}
        ]
    }
elif filter_params.keyword:
    # 키워드 조건만 있는 경우
    filter_dict = {"keyword": {filter_params.keyword_expression: filter_params.keyword}}
elif filter_params.question_id:
    # 질문 ID 조건만 있는 경우
    filter_dict = {"question_id": {filter_params.question_id_expression: filter_params.question_id}}

# retriever에 필터 적용 
retriever = vector_store.as_retriever(
    search_kwargs={"filter": filter_dict} if filter_dict else {}
)

# 검색 실행
query = "주택건설지역 관련 문서를 질문 ID 10번 이하인 문서중에서 검색해주세요"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import chain
from pydantic import BaseModel, Field
from typing import Optional, Dict


class MetadataFilter(BaseModel):
    keyword: Optional[str] = Field(description="검색할 키워드")
    keyword_expression: Optional[str] = Field(description="키워드 검색 표현식")
    question_id: Optional[int] = Field(description="질문 ID의 최소값")
    question_id_expression: Optional[str] = Field(description="질문 ID 검색 표현식")


def create_metadata_filter(query: str, llm) -> Dict:
    """
    사용자 쿼리를 분석하여 Chroma DB 검색을 위한 메타데이터 필터를 생성합니다.

    Args:
        query: 사용자 검색 쿼리.
        llm: 사용할 언어 모델.

    Returns:
        Chroma DB 검색에 사용할 수 있는 필터 딕셔너리.
    """
    system_prompt = """사용자 쿼리에서 키워드와 질문 ID 정보를 추출하여 Chroma DB 검색 필터를 생성한다.
    다음 예시를 참조한다:

    1. 키워드 검색 예시:
    - 입력: "주택건설 관련 문서 찾아줘"
    출력:
    keyword: "주택건설"
    keyword_expression: "$eq"

    2. 질문 ID 검색 예시:
    - 입력: "질문 ID 10번 이상인 문서"
    출력:
    question_id: 10
    question_id_expression: "$gte"

    3. 복합 검색 예시:
    - 입력: "주택건설 키워드가 있으면서 질문 ID가 5번에서 15번 사이인 문서"
    출력:
    keyword: "주택건설"
    keyword_expression: "$eq"
    question_id: 5
    question_id_expression: "$gte"

    검색 표현식은 다음과 같이 사용한다:
    - 동등 비교: $eq
    - 크거나 같음: $gte
    - 작거나 같음: $lte
    - 범위 검색: $gt, $lt

    요청에 해당 정보가 없는 경우 해당 필드는 null로 반환한다.
    """
    prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{query}")
    ])

    model_with_structure = llm.with_structured_output(MetadataFilter)
    metadata_chain = prompt | model_with_structure

    filter_params = metadata_chain.invoke({"query": query})

    filter_dict = {}

    if filter_params.keyword and filter_params.question_id:
        filter_dict = {
            "$and": [
                {"keyword": {filter_params.keyword_expression: filter_params.keyword}},
                {"question_id": {filter_params.question_id_expression: filter_params.question_id}}
            ]
        }
    elif filter_params.keyword:
        filter_dict = {"keyword": {filter_params.keyword_expression: filter_params.keyword}}
    elif filter_params.question_id:
        filter_dict = {"question_id": {filter_params.question_id_expression: filter_params.question_id}}

    return filter_dict



# 사용 예시 
query = "주택건설지역 관련 문서를 10번 이하인 문서중에서 검색해주세요"

@chain
def metadata_filter_query(query: str):

    llm = ChatOpenAI(
        model='gpt-4.1-mini',
        temperature=0.1,
        top_p=0.9,
    )

    filter_dict = create_metadata_filter(query, llm)
    print(filter_dict)

    retriever = vector_store.as_retriever(
            search_kwargs={"filter": filter_dict} if filter_dict else {}
    )
    results = retriever.invoke(query)

    return results

results = metadata_filter_query.invoke(query)

for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

In [ ]:
# 다른 쿼리
query = "청약통장에 대한 정보를 찾아주세요"
results = metadata_filter_query.invoke(query)

for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print(result.metadata['question_id'])
    print("=" * 50)

##### ***[실습] 메타데이터 필터링 체인의 성능을 개선합니다.*** 

- (예시)
    - 추가 예시 제공을 통해 필터링 적용 법위 확대
    - 복합조건식 추출 기능을 추가 등

In [ ]:
# 여기에 코드를 작성하세요.

# RAG Chain

### 1) 참조 문서 없이 직접 답변을 생성

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Prompt
template = '''Answer the question based only on the following context.

[Context]
{context}

[Question]
{question}

[Answer (in 한국어)]
'''

prompt = ChatPromptTemplate.from_template(template)


# 문서 포맷팅
def format_docs(docs):
    return '\n\n'.join([d.page_content for d in docs])


# 검색기 생성 - 유사도 기반 상위 3개 문서 검색
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3},
)


# Chain 구성
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Chain 실행
query = "수원시의 주택건설지역은 어디에 해당하나요?"
rag_chain.invoke(query)

### 2) 참조 문서를 답변과 함께 반환

`(1) 문서와 포맷팅된 컨텍스트를 함께 반환하는 함수`


In [ ]:
# 다음 코드를 완성하세요.

from typing import Dict

def get_context_and_docs(question: str) -> Dict:
    """문서와 포맷팅된 컨텍스트를 함께 반환
    
    Args:
        question: 검색할 질문

    Returns:
        Dict: 문서와 포맷팅된 컨텍스트, 검색된 문서 리스트
    """

    # 검색 결과 가져오기
    docs = retriever.invoke(question)
    return {
        "question": None,  # 질문
        "context": None,   # 문서 포맷팅된 컨텍스트
        "source_documents": None   # 검색된 문서 리스트
    }

`(2) 컨텍스트와 질문을 입력으로 받아 답변을 생성하는 함수`

In [ ]:
# 다음 코드를 완성하세요.

from langchain_core.output_parsers import StrOutputParser

def prompt_and_generate_answer(input_data: Dict) -> Dict:
    """컨텍스트와 질문을 입력으로 받아 답변을 생성

    Args:
        input_data (Dict): 컨텍스트와 질문이 포함된 딕셔너리

    Returns:
        Dict: 생성된 답변과 소스 문서 정보가 포함된 딕셔너리
    """

    # LCEL 체인 구성 (StrOutputParser 사용)
    answer_chain = None

    return {
        "answer": None,  # 생성된 답변 (answer_chain 결과)
        "source_documents": None  # 소스 문서 정보 (input_data에서 가져옴)
    }

`(3) RAG 체인 구성`

In [ ]:
# 다음 코드를 완성하세요.

from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from operator import itemgetter

# Chain 구성
rag_chain = (
    None(get_context_and_docs) |  # 문서와 컨텍스트 가져오기 
    {
        'response': None(prompt_and_generate_answer), # 답변 생성
        'question': None(),
        "source_documents": None("source_documents")   # 소스 반환
    }
)

In [ ]:
# Chain 실행
query = "수원시의 주택건설지역은 어디에 해당하나요?"
result = rag_chain.invoke(query)

# 결과 출력
print("답변:", result["response"]["answer"])
print("\n참조 문서:")
for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")

### 3) 검색 문서 관련성 평가

`(1) 검색 문서와 질문 간의 관련성을 평가`


In [ ]:
# 검색 문서의 질문 관련성 평가

prompt = ChatPromptTemplate.from_messages([
    ("system", """주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 논리적으로 평가하세요.
단계적으로 진행하며, 평가결과에 대한 검증을 수행하세요.

다음 기준 중 하나 이상을 충족할 경우 'Yes'로 답변하고, 모두 충족하지 못하면 'No'로 답변하세요:

1. 컨텍스트가 질문에 답변하는데 필요한 정보를 직접적으로 포함하고 있는가?
2. 컨텍스트의 정보로부터 답변에 필요한 내용을 논리적으로 추론할 수 있는가?
3. 컨텍스트의 정보가 질문에 대한 답변을 제공할 수 있는가?

'Yes' 또는 'No'로만 답변하세요."""),
    ("human", """[컨텍스트]
{context}

[질문]
{question}""")
])

chain = prompt | llm | StrOutputParser()    # gpt-4.1-mini 모델 사용

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")
    relevance = chain.invoke({
        "context": doc.page_content,
        "question": query
    }).lower()

    print(f"평가 결과: {relevance}")

In [ ]:
# gpt-4.1 모델 사용

llm_gpt4o = ChatOpenAI(
    model='gpt-4.1',
    temperature=0.1,
    top_p=0.9, 
)

chain = prompt | llm_gpt4o | StrOutputParser()    # gpt-4.1 모델 사용

for i, doc in enumerate(result["source_documents"], 1):
    print(f"\n문서 {i}:")
    print(f"내용: {doc.page_content}")
    relevance = chain.invoke({
        "context": doc.page_content,
        "question": query
    }).lower()

    print(f"평가 결과: {relevance}")

##### ***[실습] 문서 관련성 평가 체인을 구조화 출력으로 구현합니다.*** 

- pydantic schema 사용
- with_structured_output 함수 사용

In [ ]:
# 여기에 코드를 작성하세요.

# Gradio 챗봇 인터페이스

In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
vector_sotre = Chroma(
    collection_name="housing_faq_db",
    persist_directory="./chroma_db", 
    embedding_function=embeddings,
)


# 검색기 생성 - 유사도 기반 상위 3개 문서 검색
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3},
)

# 테스트 질문
query = "수원시의 주택건설지역은 어디에 해당하나요?"

results = retriever.invoke(query)
for result in results:
    print(result.page_content)
    print("-" * 50)
    print(result.metadata['keyword'])
    print("=" * 50)

In [ ]:
import gradio as gr
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStoreRetriever
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from typing import List, Optional
from dataclasses import dataclass

@dataclass
class SearchResult:
    context: str
    source_documents: Optional[List]

class RAGSystem:
    def __init__(
            self, 
            llm: BaseChatModel, 
            eval_llm: BaseChatModel,
            retriever: VectorStoreRetriever
        ):
        if not llm:
            self.llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
        else:
            self.llm = llm

        if not eval_llm:
            self.eval_llm = ChatOpenAI(model="gpt-4.1", temperature=0)
        else:
            self.eval_llm = eval_llm

        if not retriever:
            raise ValueError("검색기(retriever)가 필요합니다.")
        else:
            self.retriever = retriever
        
    def _format_docs(self, docs: List) -> str:
        return "\n\n".join(doc.page_content for doc in docs)
    
    def _format_source_documents(self, docs: Optional[List]) -> str:
        if not docs:
            return "\n\nℹ️ 관련 문서를 찾을 수 없습니다."
        
        formatted_docs = []
        for i, doc in enumerate(docs, 1):
            metadata = doc.metadata if hasattr(doc, 'metadata') else {}
            source_info = []
            
            if 'question_id' in metadata:
                source_info.append(f"ID: {metadata['question_id']}")
            if 'keyword' in metadata:
                source_info.append(f"키워드: {metadata['keyword']}")
            if 'summary' in metadata:
                source_info.append(f"요약: {metadata['summary']}")
                
            formatted_docs.append(
                f"📚 참조 문서 {i}\n"
                f"• {' | '.join(source_info) if source_info else '출처 정보 없음'}\n"
                f"• 내용: {doc.page_content}"
            )
        
        return "\n\n" + "\n\n".join(formatted_docs)
    
    def _check_relevance(self, docs: List, question: str) -> List:
        """문서의 관련성 확인"""

        relevant_docs = []

        if not docs:
            return relevant_docs
            
        prompt = ChatPromptTemplate.from_messages([
            ("system", """주어진 컨텍스트가 질문에 답변하는데 필요한 정보를 포함하고 있는지 평가하세요.

        다음 기준 중 하나 이상을 충족할 경우 'Yes'로 답변하고, 모두 충족하지 못하면 'No'로 답변하세요:

        1. 컨텍스트가 질문에 답변하는데 필요한 정보를 직접적으로 포함하고 있는가?
        2. 컨텍스트의 정보로부터 답변에 필요한 내용을 논리적으로 추론할 수 있는가?

        'Yes' 또는 'No'로만 답변하세요."""),
            ("human", """[컨텍스트]
        {context}

        [질문]
        {question}""")
        ])
        
        chain = prompt | self.eval_llm | StrOutputParser()

        for doc in docs:
            result = chain.invoke({
                "context": doc.page_content,
                "question": question
            }).lower()

            print(f"문서 {doc.metadata['question_id']} 관련성 확인 결과: {result}")
            print(f"문서 {doc.metadata['question_id']} 내용:")
            print(doc.page_content)
            print("-" * 50)

            if "yes" in result:
                relevant_docs.append(doc)
            
        return relevant_docs
    
    def search_documents(self, question: str) -> SearchResult:
        try:
            docs = retriever.invoke(question)
            print(f"검색된 문서 개수: {len(docs)}")
            relevant_docs = self._check_relevance(docs, question) 
            print(f"관련 문서 개수: {len(relevant_docs)}")
            
            return SearchResult(
                context=self._format_docs(relevant_docs) if relevant_docs else "관련 문서를 찾을 수 없습니다.",
                source_documents=relevant_docs,
            )
        except Exception as e:
            print(f"문서 검색 중 오류 발생: {e}")
            return SearchResult(
                context="문서 검색 중 오류가 발생했습니다.",
                source_documents=None,
            )
    
    def generate_answer(self, message: str, history: List) -> str:
        # 문서 검색
        search_result = self.search_documents(message)
        
        if not search_result.source_documents:
            return "죄송합니다. 관련 문서를 찾을 수 없어 답변하기 어렵습니다. 다른 질문을 해주시겠습니까?"
                    
        # 프롬프트 템플릿 설정
        prompt = ChatPromptTemplate.from_messages([
            ("system", """다음 지침을 따라 질문에 답변해주세요:
            1. 주어진 문서의 내용만을 기반으로 답변하세요.
            2. 문서에 명확한 근거가 없는 내용은 "근거 없음"이라고 답변하세요.
            3. 답변하기 어려운 질문은 "잘 모르겠습니다"라고 답변하세요.
            4. 추측이나 일반적인 지식을 사용하지 마세요."""),
            ("human", "문서들:\n{context}\n\n질문: {question}")
        ])
        
        # RAG Chain 구성
        chain = prompt | self.llm | StrOutputParser()
        
        try:
            # 답변 생성
            answer = chain.invoke({
                "context": search_result.context,
                "question": message
            })
            
            # 참조 문서 포맷팅 추가
            sources = self._format_source_documents(search_result.source_documents)
            return f"{answer}\n{sources}"
            
        except Exception as e:
            return f"답변 생성 중 오류가 발생했습니다: {str(e)}"

# Gradio 인터페이스 설정

rag_system = RAGSystem(
    llm=ChatOpenAI(model="gpt-4.1-mini", temperature=0),   
    eval_llm=ChatOpenAI(model="gpt-4.1", temperature=0),
    retriever=vector_store.as_retriever(search_kwargs={"k": 3})
)

demo = gr.ChatInterface(
    fn=rag_system.generate_answer,
    title="RAG QA 시스템",
    description="""
    질문을 입력하면 관련 문서를 검색하여 답변을 생성합니다.
    모든 답변에는 참조한 문서의 출처가 표시됩니다.
    """,
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="gray",
    ),
examples=[
    ["수원시의 주택건설지역은 어디에 해당하나요?"],
    ["무주택 세대에 대해서 설명해주세요."],
    ["2순위로 당첨된 사람이 청약통장을 다시 사용할 수 있나요?"],
],
)

# 데모 실행
demo.launch()

In [ ]:
# Gradio 인터페이스 종료
demo.close()

---

# **[실습] 주택청약 FAQ 시스템 구현**

### **문제 설명**
이전 코드를 기반으로 주택청약 FAQ 시스템을 다음 요구사항에 맞춰 개선합니다. 

1. 응답 품질 향상 (1개 이상)
   - 생성된 답변의 품질을 평가 (답변이 불충분한 경우 예외 처리)
   - 관련성 높은 FAQ 문서 검색 (임베딩 모델, 청크 크기, 벡터 검색 방법 등)

2. 사용자 경험 개선 (1개 이상)
   - 대화 이력 관리 기능 추가 (요약, 트리밍 기능 등 고려)
   - 최근 대화 기반 컨텍스트 구성 
   - 사용자 프로필 기반 맞춤 응답

### **제약 조건**
- Gradio ChatInterface 사용
- RAG 구조 유지

In [45]:
# 표준 라이브러리
import re
from glob import glob
from dataclasses import dataclass
from typing import Dict, List

# 환경변수 관리
from dotenv import load_dotenv

# 웹 인터페이스
import gradio as gr

# 데이터 모델링
from pydantic import BaseModel, Field

# LLM 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 문서 처리
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

# 프롬프트 및 출력 처리
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import tiktoken

# 벡터 저장소
from langchain_chroma import Chroma

# 환경변수 로드
load_dotenv()

# LLM 초기화
llm = ChatOpenAI(
    model='gpt-4.1-mini',
    temperature=0.1,
    top_p=0.9, 
)

# 데이터 파일 로드
data_files = glob("../data/housing_faq.txt")
loader = TextLoader(data_files[0], 'utf-8')
docs = loader.load()

# 문서 전처리 함수
def extract_qa_pairs(text):
    qa_pairs = []
    
    # 텍스트를 라인별로 분리하고 각 라인의 앞뒤 공백 제거
    lines = [line.strip() for line in text.split('\n')]
    current_question = None
    current_answer = []
    current_number = None
    in_answer = False
    
    for i, line in enumerate(lines):
        if not line:  # 빈 라인 처리
            if in_answer and current_answer and i + 1 < len(lines) and lines[i + 1].startswith('Q'):
                # 다음 질문이 시작되기 전 빈 줄이면 현재 QA 쌍 저장
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
                in_answer = False
                current_answer = []
            continue
            
        # 새로운 질문 확인 (Q 다음에 숫자가 오는 패턴)
        q_match = re.match(r'Q(\d+)\s+(.*)', line)
        if q_match:
            # 이전 QA 쌍이 있으면 저장
            if current_question is not None and current_answer:
                qa_pairs.append({
                    'number': current_number,
                    'question': current_question,
                    'answer': ' '.join(current_answer).strip()
                })
            
            # 새로운 질문 시작
            current_number = int(q_match.group(1))
            current_question = q_match.group(2).strip().rstrip('?') + '?'  # 질문 마크 정규화
            current_answer = []
            in_answer = False
            
        # 답변 시작 확인
        elif line.startswith('A ') or (current_question and not current_answer and line):
            in_answer = True
            current_answer.append(line.lstrip('A '))
            
        # 기존 답변에 내용 추가
        elif current_question is not None and (in_answer or not line.startswith('Q')):
            if in_answer or (current_answer and not line.startswith('Q')):
                current_answer.append(line)
    
    # 마지막 QA 쌍 처리
    if current_question is not None and current_answer:
        qa_pairs.append({
            'number': current_number,
            'question': current_question,
            'answer': ' '.join(current_answer).strip()
        })
    
    # 번호 순서대로 정렬
    qa_pairs.sort(key=lambda x: x['number'])
    
    return qa_pairs

# 응답 품질 평가 모델
class ResponseQuality(BaseModel):
    relevance_score: float = Field(description="답변의 관련성 점수 (0-1)")
    completeness_score: float = Field(description="답변의 완성도 점수 (0-1)")
    confidence_score: float = Field(description="답변의 신뢰도 점수 (0-1)")
    overall_quality: str = Field(description="전체 품질 평가 (excellent/good/fair/poor)")
    improvement_suggestions: str = Field(description="개선 제안사항")

# 사용자 프로필 모델
@dataclass
class UserProfile:
    interests: List[str]
    experience_level: str
    recent_topics: List[str]
    preferred_detail_level: str

# 대화 이력 관리
class ConversationManager:
    def __init__(self, max_history: int = 10, max_tokens: int = 2000):
        self.max_history = max_history
        self.max_tokens = max_tokens
        self.conversation_history: List[Dict[str, str]] = []
        self.user_profiles: Dict[str, UserProfile] = {}
        self.summarized_history: Dict[str, str] = {}  # 사용자별 요약된 이력
        
        # 토큰 계산기
        try:
            self.encoding = tiktoken.encoding_for_model("gpt-4o-mini")
        except:
            self.encoding = tiktoken.get_encoding("cl100k_base")

    def count_tokens(self, text: str) -> int:
        """텍스트의 토큰 수 계산"""
        return len(self.encoding.encode(text))
    
    def summarize_old_conversations(self, user_id: str, conversations_to_summarize: List[Dict]) -> str:
        """오래된 대화들을 요약"""
        if not conversations_to_summarize:
            return ""
        
        summary_text = ""
        for conv in conversations_to_summarize:
            summary_text += f"Q: {conv['question']}\nA: {conv['answer'][:100]}...\n\n"
        
        # LLM을 사용한 요약
        summary_prompt = ChatPromptTemplate.from_messages([
            ("system", "다음 주택청약 관련 대화들을 핵심 내용만 간단히 요약해주세요. 사용자가 관심있어 했던 주제와 주요 질문들을 중심으로 3-4줄로 정리하세요."),
            ("user", f"요약할 대화:\n{summary_text}")
        ])
        
        try:
            summary_chain = summary_prompt | llm | StrOutputParser()
            summary = summary_chain.invoke({})
            return summary
        except:
            return "이전 대화에서 청약 관련 다양한 질문들이 있었습니다."
    
    def add_exchange(self, user_id: str, question: str, answer: str, topics: List[str] = None):
        exchange = {
            "user_id": user_id,
            "question": question,
            "answer": answer,
            "topics": topics or [],
            "timestamp": "now"
        }
        self.conversation_history.append(exchange)
        
        # 토큰 기반 관리
        self.manage_history_by_tokens(user_id)
        
        # 기존 프로필 업데이트
        self.update_user_profile(user_id, topics or [])

    def manage_history_by_tokens(self, user_id: str):
        """토큰 수 기반으로 이력 관리"""
        user_conversations = [h for h in self.conversation_history if h["user_id"] == user_id]
        
        # 현재 대화들의 총 토큰 수 계산
        total_tokens = 0
        for conv in user_conversations:
            total_tokens += self.count_tokens(conv['question'] + conv['answer'])
        
        # 토큰 수가 초과하면 처리
        if total_tokens > self.max_tokens or len(user_conversations) > self.max_history:
            # 오래된 대화 절반을 요약
            conversations_to_summarize = user_conversations[:len(user_conversations)//2]
            
            if conversations_to_summarize:
                # 요약 생성
                new_summary = self.summarize_old_conversations(user_id, conversations_to_summarize)
                
                # 기존 요약과 합치기
                if user_id in self.summarized_history:
                    self.summarized_history[user_id] = f"{self.summarized_history[user_id]}\n\n{new_summary}"
                else:
                    self.summarized_history[user_id] = new_summary
                
                # 요약된 대화들을 실제 이력에서 제거
                for conv in conversations_to_summarize:
                    if conv in self.conversation_history:
                        self.conversation_history.remove(conv)
    
    def update_user_profile(self, user_id: str, topics: List[str]):
        if user_id not in self.user_profiles:
            self.user_profiles[user_id] = UserProfile(
                interests=[],
                experience_level="beginner",
                recent_topics=[],
                preferred_detail_level="detailed"
            )
        
        profile = self.user_profiles[user_id]
        profile.recent_topics.extend(topics)
        profile.recent_topics = profile.recent_topics[-5:]  # 최근 5개만 유지
    
    def get_context(self, user_id: str) -> str:
        context_parts = []
        
        # 요약된 이전 대화
        if user_id in self.summarized_history and self.summarized_history[user_id]:
            context_parts.append(f"이전 대화 요약:\n{self.summarized_history[user_id]}")
        
        # 최근 상세 대화 (최근 5개)
        recent_history = [h for h in self.conversation_history[-3:] if h["user_id"] == user_id]
        if recent_history:
            recent_context = "최근 대화:\n"
            for i, h in enumerate(recent_history, 1):
                recent_context += f"{i}. Q: {h['question']}\n   A: {h['answer'][:100]}...\n\n"
            context_parts.append(recent_context)
        
        return "\n\n---\n\n".join(context_parts)

    def get_user_profile(self, user_id: str) -> UserProfile:
        """사용자 프로필 조회"""
        if user_id not in self.user_profiles:
            self.user_profiles[user_id] = UserProfile(
                interests=["주택청약", "분양"],
                experience_level="beginner",
                recent_topics=[],
                preferred_detail_level="detailed"
            )
        return self.user_profiles[user_id]

# QA 쌍 추출
qa_pairs = extract_qa_pairs(docs[0].page_content) 

# Document 객체로 변환
formatted_docs = []
for qa in qa_pairs:
    doc_content = f"질문: {qa['question']}\n답변: {qa['answer']}"
    doc = Document(
        page_content=doc_content,
        metadata={
            "question": qa['question'],
            "answer": qa['answer'],
            "number": qa['number']
        }
    )
    formatted_docs.append(doc)

# 임베딩 모델
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터 저장소 로드
vector_store = Chroma.from_documents(  
    documents=formatted_docs,
    embedding=embeddings,
    collection_name="housing_faq_db",
    persist_directory="./chroma_db",
)

# 검색기 생성
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# 응답 품질 평가 체인
quality_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 답변 품질을 평가하는 전문가입니다.
주어진 질문과 답변을 분석하여 다음 기준으로 평가해주세요:
- 관련성: 답변이 질문과 얼마나 관련이 있는가
- 완성도: 답변이 얼마나 완전한가
- 신뢰도: 답변이 얼마나 신뢰할 만한가"""),
    ("user", "질문: {question}\n답변: {answer}\n\n위 답변을 평가해주세요.")
])

quality_evaluator = quality_prompt | llm.with_structured_output(ResponseQuality)

# RAG 체인
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 주택청약 전문 상담사입니다.
제공된 FAQ 정보를 바탕으로 사용자의 질문에 정확하고 도움이 되는 답변을 제공하세요.
     
**중요**: 질문이 주택청약과 관련이 없다면 정중히 안내하고 주택청약 관련 질문을 요청하세요.

주택청약 관련 질문(청약통장, 분양, 당첨 조건, 전매 등)에만 답변하고,
그 외 질문에는 "주택청약 관련 질문에만 답변드릴 수 있습니다"라고 응답하세요.

**사용자 프로필:**
- 경험 수준: {experience_level}
- 관심 분야: {interests}
- 선호 설명 수준: {detail_level}

**답변 가이드라인:**
- 사용자의 경험 수준에 맞춰 설명하세요
- 정확한 정보를 제공하되, 이해하기 쉽게 설명하세요
- 관련 법령이나 정책이 변경될 수 있음을 언급하세요
- 구체적인 사례나 예시를 포함하여 설명하세요
- 추가 문의가 필요한 경우 관련 기관을 안내하세요

**컨텍스트 정보:**
{context}

**사용자 대화 이력:**
{user_context}"""),
    
    ("user", "{question}")
])

conversation_manager = ConversationManager()

# 답변 생성 함수
def answer_question(question: str, user_id: str = "default_user") -> str:
    try:
        # 관련 문서 검색
        relevant_docs = retriever.get_relevant_documents(question)
        
        # 컨텍스트 구성
        context = "\n\n".join([doc.page_content for doc in relevant_docs])
        user_context = conversation_manager.get_context(user_id)
        
        # 사용자 프로필 정보 가져오기
        user_profile = conversation_manager.get_user_profile(user_id)
        
        # 답변 생성
        rag_chain = rag_prompt | llm | StrOutputParser()
        answer = rag_chain.invoke({
            "question": question,
            "context": context,
            "user_context": user_context,
            "experience_level": user_profile.experience_level,
            "interests": ", ".join(user_profile.interests) if user_profile.interests else "주택청약 일반",
            "detail_level": user_profile.preferred_detail_level
        })
        
        # 품질 평가
        try:
            quality_result = quality_evaluator.invoke({
                "question": question,
                "answer": answer
            })
            
            # 답변이 너무 부족한 경우 경고 추가
            if quality_result.get("relevance_score", 0.8) < 0.4:
                answer += "\n\n💡 더 구체적인 질문을 해주시면 더 정확한 답변을 드릴 수 있습니다."
                
        except Exception as e:
            print(f"품질 평가 오류: {e}")
        
        # 대화 이력에 추가
        conversation_manager.add_exchange(user_id, question, answer)
        
        return answer
        
    except Exception as e:
        return f"죄송합니다. 답변 생성 중 오류가 발생했습니다: {str(e)}"


def chat_interface(message, history):
    "Gradio 채팅 인터페이스"
    user_id = "default_user"
    
    # History 동기화 함수 호출
    sync_gradio_history_to_manager(history, user_id)

    # answer_question 함수 호출
    response = answer_question(message, user_id)
    
    return response

# History 동기화 함수
def sync_gradio_history_to_manager(gradio_history, user_id: str):
    
    # 현재 ConversationManager의 이력 길이
    current_manager_length = len([h for h in conversation_manager.conversation_history if h["user_id"] == user_id])
    
    # Gradio history가 더 길면 동기화 필요
    if len(gradio_history) > current_manager_length:
        # 새로운 대화만 추가
        for i in range(current_manager_length, len(gradio_history)):
            if i < len(gradio_history):
                user_msg, assistant_msg = gradio_history[i]
                if user_msg and assistant_msg:  # 둘 다 존재할 때만 추가
                    conversation_manager.add_exchange(
                        user_id=user_id,
                        question=user_msg,
                        answer=assistant_msg,
                        topics=[]
                    )

def create_gradio_interface():
    """Gradio 인터페이스 생성"""
    with gr.Blocks(title="주택청약 FAQ 챗봇") as demo:
        gr.Markdown("""
        # 🏠 주택청약 FAQ 챗봇
        
        주택청약에 관한 궁금한 점을 물어보세요!
        - 청약통장, 청약 조건, 분양 정보 등
        - 대화 이력을 바탕으로 맞춤형 답변 제공
        - 답변 품질을 실시간으로 평가
        - 사용자 프로필 기반 개인화 답변
        """)
        
        # 채팅 인터페이스
        chatbot = gr.ChatInterface(
            fn=chat_interface,
            title="",
            description="",
            examples=[
                "청약통장이란 무엇인가요?",
                "1순위 청약 조건을 알려주세요",
                "분양권 전매는 언제부터 가능한가요?",
                "청약 당첨 확률을 높이는 방법이 있나요?"
            ],
            submit_btn="질문하기",
        )
        
        # 시스템 정보
        with gr.Accordion("시스템 정보", open=False):
            gr.Markdown(f"""
            - **벡터 DB**: {len(formatted_docs)}개 FAQ 문서 저장됨
            - **검색 방식**: 유사도 기반 상위 3개 문서 검색
            - **대화 이력**: 최근 10개 대화 저장
            - **품질 평가**: 관련성, 완성도, 신뢰도 자동 평가
            - **개인화**: 사용자 프로필 기반 맞춤형 답변
            """)
    
    return demo

# 메인 실행
if __name__ == "__main__":    
    # Gradio 앱 실행
    demo = create_gradio_interface()
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=False,
        debug=True
    )

e:\study\modulab-ai\prj01\faq_bot\.venv\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


품질 평가 오류: 'ResponseQuality' object has no attribute 'get'
품질 평가 오류: 'ResponseQuality' object has no attribute 'get'
품질 평가 오류: 'ResponseQuality' object has no attribute 'get'
Keyboard interruption in main thread... closing server.
